# Global Firepower 2026
## Data Collection & Preparation Pipeline

### Project purpose



This notebook documents the **data collection, cleaning, transformation, validation, and export** stages of the Global Firepower 2026 analysis project.

The project combines country-level indicators across eight dimensions:

- Manpower
- Finance
- Air Force
- Army
- Navy
- Logistics
- Geography
- Natural Resources

The resulting datasets are saved as analysis-ready `.pkl` files and are used by the separate **EDA / Business Analysis notebook**.



## 1. Project workflow

```text
Global Firepower source pages
            ↓
     Selenium extraction
            ↓
     Country-level datasets
            ↓
 Cleaning & type conversion
            ↓
 Unit standardization
            ↓
 Column standardization
            ↓
 Data-quality validation
            ↓
 8 analysis-ready PKL files
            ↓
 Global Firepower EDA notebook
```

### Analytical handoff

This notebook answers:

> **How was the analysis-ready data collected and prepared?**

The EDA notebook answers:

> **What can we learn from the prepared data?**

## 2. Data sources

The project uses country-level tables from the **Global Firepower** website. The original project collected the following categories:

| Dataset | Purpose |
|---|---|
| Manpower | Population and military manpower indicators plus overall Power Index |
| Finance | Defense budget and broader financial indicators |
| Air Force | Aircraft fleet composition |
| Army | Ground-force equipment |
| Navy | Naval fleet composition and tonnage |
| Logistics | Airports, ports, merchant marine, railways and roads |
| Geography | Land area, coastline, borders and waterways |
| Natural Resources | Oil, natural gas and coal indicators |

The source pages are accessed through Selenium because the original collection process used dynamically rendered tables.

**Source:** https://www.globalfirepower.com/

> Re-running the scraper may return updated source values if the website changes. The submitted `.pkl` files represent the prepared project snapshot used for the analysis.

## 3. Libraries and configuration

In [1]:
from pathlib import Path
import re
import time

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# -------------------------------------------------------------------
# Project configuration
# -------------------------------------------------------------------

DATA_DIR = Path(".")
DATA_YEAR = 2026
SCRAPE_TIMEOUT = 10

# Fresh 2026 collection run: scrape the current source pages and build new 2026 datasets.
RUN_SCRAPING = True

DATASETS = [
    "df_manpower_2026",
    "df_finance_2026",
    "df_airforce_2026",
    "df_army_2026",
    "df_navy_2026",
    "df_logistics_2026",
    "df_geography_2026",
    "df_nat_resources_2026",
]

print("Configured datasets:", len(DATASETS))

Configured datasets: 8


## 4. Reusable Selenium collection functions

The scraper:

1. Opens a Global Firepower page.
2. Waits for the country-level records to load.
3. Extracts country and value fields.
4. Cleans text such as commas, currency symbols and units.
5. Converts extracted values to numeric where appropriate.
6. Merges additional metrics using `country` as the common key.

In [2]:
def create_driver():
    """Create a Chrome WebDriver using webdriver-manager."""
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install())
    )


def clean_numeric(value):
    """Remove common display characters and convert a value to numeric."""
    if value is None:
        return None

    text = str(value).strip()
    text = re.sub(r"[$,]", "", text)
    text = re.sub(r"\b(?:km|bbl|Cu\.M|mt)\b", "", text, flags=re.IGNORECASE)
    text = text.replace("PwrIndx:", "").strip()

    return pd.to_numeric(text, errors="coerce")


def scrape_ranked_page(driver, url, value_column, include_rank=True, timeout=SCRAPE_TIMEOUT):
    """Extract country-level values from a Global Firepower ranked table."""
    driver.get(url)

    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.CLASS_NAME, "recordsetContainer"))
    )

    records = []

    for block in driver.find_elements(By.CLASS_NAME, "recordsetContainer"):
        try:
            country = block.find_element(
                By.CLASS_NAME, "longFormName"
            ).text.strip()

            value = block.find_element(
                By.CLASS_NAME, "valueContainer"
            ).text.strip()

            record = {
                "country": country,
                value_column: clean_numeric(value),
            }

            if include_rank:
                rank = block.find_element(
                    By.CLASS_NAME, "rankNumContainer"
                ).text.strip()
                record["rank"] = pd.to_numeric(rank, errors="coerce")

            records.append(record)

        except Exception:
            # Skip malformed records rather than stopping the full collection.
            continue

    result = pd.DataFrame(records)

    if include_rank:
        result = result[["rank", "country", value_column]]
    else:
        result = result[["country", value_column]]

    return result


def scrape_metrics(driver, base_df, metric_urls, timeout=SCRAPE_TIMEOUT):
    """Add multiple country-level metrics to an existing dataset."""
    result = base_df.copy()

    for metric_name, url in metric_urls.items():
        metric_df = scrape_ranked_page(
            driver,
            url,
            value_column=metric_name,
            include_rank=False,
            timeout=timeout,
        )
        result = result.merge(metric_df, on="country", how="left")

    return result


def close_driver(driver):
    if driver is not None:
        driver.quit()

## 5. Source-page configuration

The URLs below reproduce the source-page structure.

In [3]:
SOURCE_URLS = {
    "overall_power": "https://www.globalfirepower.com/countries-listing.php",

    "manpower": {
        "available_manpower": "https://www.globalfirepower.com/available-military-manpower.php",
        "paramilitary": "https://www.globalfirepower.com/manpower-paramilitary.php",
        "labor_force_strength": "https://www.globalfirepower.com/labor-force-by-country.php",
        "active_service": "https://www.globalfirepower.com/active-military-manpower.php",
        "active_reserves": "https://www.globalfirepower.com/active-reserve-military-manpower.php",
    },

    "finance": {
        "annual_defense_budget": "https://www.globalfirepower.com/defense-spending-budget.php",
        "external_debt": "https://www.globalfirepower.com/external-debt-by-country.php",
        "purchasing_power_parity": "https://www.globalfirepower.com/purchasing-power-parity.php",
        "foreign_exchange_reserves": "https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php",
        "fit_for_service": "https://www.globalfirepower.com/manpower-fit-for-military-service.php",
    },

    "logistics": {
        "serviceable_airports": "https://www.globalfirepower.com/major-serviceable-airports-by-country.php",
        "major_ports_terminals": "https://www.globalfirepower.com/major-ports-and-terminals.php",
        "merchant_marine_strength": "https://www.globalfirepower.com/merchant-marine-strength-by-country.php",
        "railway_coverage": "https://www.globalfirepower.com/railway-coverage.php",
        "roadway_coverage": "https://www.globalfirepower.com/roadway-coverage.php",
    },

    "natural_resources": {
        "oil_production": "https://www.globalfirepower.com/oil-production-by-country.php",
        "oil_consumption": "https://www.globalfirepower.com/oil-consumption-by-country.php",
        "oil_reserves": "https://www.globalfirepower.com/proven-oil-reserves-by-country.php",
        "natural_gas_production": "https://www.globalfirepower.com/natural-gas-production-by-country.php",
        "natural_gas_consumption": "https://www.globalfirepower.com/natural-gas-consumption-by-country.php",
        "natural_gas_reserves": "https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php",
        "coal_production": "https://www.globalfirepower.com/coal-production-by-country.php",
        "coal_consumption": "https://www.globalfirepower.com/coal-consumption-by-country.php",
        "coal_reserves": "https://www.globalfirepower.com/proven-coal-reserves-by-country.php",
    },

    "geography": {
        "coastline_coverage": "https://www.globalfirepower.com/coastline-coverage.php",
        "border_coverage": "https://www.globalfirepower.com/border-coverage.php",
        "waterway_coverage": "https://www.globalfirepower.com/waterway-coverage.php",
    },

    "airforce": {
        "total_aircraft": "https://www.globalfirepower.com/aircraft-total.php",
        "fighter_fleet": "https://www.globalfirepower.com/aircraft-total-fighters.php",
        "attack_fleet": "https://www.globalfirepower.com/aircraft-total-attack-types.php",
        "transport_fleet": "https://www.globalfirepower.com/aircraft-total-transports.php",
        "trainer_fleet": "https://www.globalfirepower.com/aircraft-total-trainers.php",
        "attack_helicopter_fleet": "https://www.globalfirepower.com/aircraft-helicopters-attack.php",
        "tanker_fleet": "https://www.globalfirepower.com/aircraft-total-tanker-fleet.php",
        "helicopter_fleet": "https://www.globalfirepower.com/aircraft-helicopters-total.php",
    },

    "army": {
        "tank_fleet_strength": "https://www.globalfirepower.com/armor-tanks-total.php",
        "armored_fighting_vehicle": "https://www.globalfirepower.com/armor-apc-total.php",
        "self_propelled_artillery": "https://www.globalfirepower.com/armor-self-propelled-guns-total.php",
        "towed_artillery": "https://www.globalfirepower.com/armor-towed-artillery-total.php",
        "rocket_projectors": "https://www.globalfirepower.com/armor-mlrs-total.php",
    },

    "navy": {
        "naval_fleet_strength": "https://www.globalfirepower.com/navy-ships.php",
        "navy_force_by_tonnage": "https://www.globalfirepower.com/navy-force-by-tonnage.php",
        "aircraft_carriers": "https://www.globalfirepower.com/navy-aircraft-carriers.php",
        "helicopter_carriers": "https://www.globalfirepower.com/navy-helo-carriers.php",
        "submarines": "https://www.globalfirepower.com/navy-submarines.php",
        "destroyers": "https://www.globalfirepower.com/navy-destroyers.php",
        "frigates": "https://www.globalfirepower.com/navy-frigates.php",
        "corvettes": "https://www.globalfirepower.com/navy-corvettes.php",
        "coastal_patrol": "https://www.globalfirepower.com/navy-patrol-coastal-craft.php",
        "mine_warfare": "https://www.globalfirepower.com/navy-mine-warfare-craft.php",
    },
}

## 6. Overall military power and base country list

The overall country listing provides the common country key, Global Firepower rank, and Power Index. This becomes the foundation for the manpower dataset and establishes the primary outcome variables used later in the EDA.

In [4]:
def collect_overall_power(driver):
    driver.get(SOURCE_URLS["overall_power"])

    WebDriverWait(driver, SCRAPE_TIMEOUT).until(
        EC.presence_of_element_located((By.CLASS_NAME, "recordsetContainer"))
    )

    records = []

    for block in driver.find_elements(By.CLASS_NAME, "recordsetContainer"):
        try:
            rank = block.find_element(By.CLASS_NAME, "rankNumContainer").text.strip()
            country = block.find_element(By.CLASS_NAME, "longFormName").text.strip()
            power_index = block.find_element(
                By.CLASS_NAME, "pwrIndxContainer"
            ).text.replace("PwrIndx:", "").strip()

            records.append({
                "rank": pd.to_numeric(rank, errors="coerce"),
                "country": country,
                "power_index": pd.to_numeric(power_index, errors="coerce"),
            })
        except Exception:
            continue

    return pd.DataFrame(records, columns=["rank", "country", "power_index"])


if RUN_SCRAPING:
    driver = create_driver()
    try:
        df_manpower_2026 = collect_overall_power(driver)
    finally:
        close_driver(driver)
else:
    df_manpower_2026 = pd.read_pickle(DATA_DIR / "df_manpower_2026.pkl")[[
        "rank", "country", "power_index"
    ]].copy()

print(df_manpower_2026.shape)
df_manpower_2026.head()

(145, 3)


,rank,country,power_index
0,1,United States,0.0741
1,2,Russia,0.0791
2,3,China,0.0919
3,4,India,0.1346
4,5,South Korea,0.1642


In [7]:
df_manpower_2026.to_pickle("df_manpower)

## 7. Manpower data collection

Manpower indicators describe the available human-resource base supporting military capability. The project combines the overall Power Index dataset with available manpower, fit-for-service population, paramilitary personnel, labor force, active service and active reserves.

In [8]:
if RUN_SCRAPING:
    # Start from the prepared project snapshot so that fields whose original
    # collection step is not documented in the development notebook are preserved.
    df_manpower_2026 = pd.read_pickle(DATA_DIR / "df_manpower_2026.pkl").copy()

    driver = create_driver()
    try:
        manpower_urls = {
            "available_manpower_million": SOURCE_URLS["manpower"]["available_manpower"],
            "paramilitary_million": SOURCE_URLS["manpower"]["paramilitary"],
            "labor_force_strength_million": SOURCE_URLS["manpower"]["labor_force_strength"],
            "active_manpower_million": SOURCE_URLS["manpower"]["active_service"],
            "active_reserves_million": SOURCE_URLS["manpower"]["active_reserves"],
            "fit_for_service_million": SOURCE_URLS["finance"]["fit_for_service"],
        }

        for metric, url in manpower_urls.items():
            metric_df = scrape_ranked_page(
                driver, url, metric, include_rank=False
            )
            # Replace a project column only when the scraped metric is available.
            df_manpower_2026 = df_manpower_2026.drop(columns=[metric], errors="ignore")
            df_manpower_2026 = df_manpower_2026.merge(metric_df, on="country", how="left")
    finally:
        close_driver(driver)
else:
    df_manpower_2026 = pd.read_pickle(DATA_DIR / "df_manpower_2026.pkl").copy()

df_manpower_2026.head()

,rank,country,power_index,available_manpower_million,paramilitary_million,labor_force_strength_million,active_manpower_million,active_reserves_million,fit_for_service_million
0,1,United States,0.0741,150463900,0,174174000,1333030,799500,124816644
1,2,Russia,0.0791,69002197,250000,72517000,1320000,2000000,46189226
2,3,China,0.0919,764123366,625000,773880000,2035000,510000,626864169
3,4,India,0.1346,662290299,2527000,607691000,1431000,1000000,522786598
4,5,South Korea,0.1642,26040900,120000,29713000,450000,3100000,21353538


## 8. Finance data collection

Financial indicators are collected to support later analysis of defense expenditure and the broader economic context.

The final project snapshot contains:

- Annual defense budget
- External debt
- Purchasing power parity
- Foreign-exchange reserves

In [9]:
if RUN_SCRAPING:
    driver = create_driver()
    try:
        finance_urls = {
            "annual_defense_budget": SOURCE_URLS["finance"]["annual_defense_budget"],
            "external_debt": SOURCE_URLS["finance"]["external_debt"],
            "purchasing_power_parity": SOURCE_URLS["finance"]["purchasing_power_parity"],
            "foreign_exchange_reserves": SOURCE_URLS["finance"]["foreign_exchange_reserves"],
        }

        df_finance_2026 = None
        for metric, url in finance_urls.items():
            metric_df = scrape_ranked_page(
                driver, url, metric, include_rank=(metric == "annual_defense_budget")
            )
            if df_finance_2026 is None:
                df_finance_2026 = metric_df
            else:
                df_finance_2026 = df_finance_2026.merge(metric_df, on="country", how="left")
    finally:
        close_driver(driver)
else:
    df_finance_2026 = pd.read_pickle(DATA_DIR / "df_finance_2026.pkl").copy()

df_finance_2026.head()

,rank,country,annual_defense_budget,external_debt,purchasing_power_parity,foreign_exchange_reserves
0,1,United States,831500000000,24538900710000,25676000000000,910037000000
1,2,China,303000000000,488114000000,33598000000000,3456000000000
2,3,Russia,212638272000,226475750000,6089000000000,597217000000
3,4,Germany,127400090000,6862470230000,5247000000000,377936000000
4,5,India,109000000000,212728000000,14244000000000,643043000000


## 9. Logistics data collection

In [10]:
if RUN_SCRAPING:
    driver = create_driver()
    try:
        logistics_urls = SOURCE_URLS["logistics"]

        df_logistics_2026 = None
        for metric, url in logistics_urls.items():
            metric_df = scrape_ranked_page(
                driver, url, metric, include_rank=(metric == "serviceable_airports")
            )
            if df_logistics_2026 is None:
                df_logistics_2026 = metric_df
            else:
                df_logistics_2026 = df_logistics_2026.merge(metric_df, on="country", how="left")
    finally:
        close_driver(driver)
else:
    df_logistics_2026 = pd.read_pickle(DATA_DIR / "df_logistics_2026.pkl").copy()

df_logistics_2026.head()

,rank,country,serviceable_airports,major_ports_terminals,merchant_marine_strength,railway_coverage,roadway_coverage
0,1,United States,16116,666,3533,293564,6586610
1,2,Brazil,5297,45,888,29850,2000000
2,3,Australia,2257,66,604,32606,873573
3,4,Mexico,1580,35,674,23389,704884
4,5,Canada,1425,284,716,49422,1042300


## 10. Natural resources data collection

In [12]:
if RUN_SCRAPING:
    driver = create_driver()
    try:
        resource_urls = SOURCE_URLS["natural_resources"]

        df_nat_resources_2026 = None
        for metric, url in resource_urls.items():
            metric_df = scrape_ranked_page(
                driver, url, metric, include_rank=(metric == "oil_production")
            )
            if df_nat_resources_2026 is None:
                df_nat_resources_2026 = metric_df
            else:
                df_nat_resources_2026 = df_nat_resources_2026.merge(metric_df, on="country", how="left")
    finally:
        close_driver(driver)
else:
    df_nat_resources_2026 = pd.read_pickle(DATA_DIR / "df_nat_resources_2026.pkl").copy()

df_nat_resources_2026.head()

,rank,country,oil_production,oil_consumption,oil_reserves,natural_gas_production,natural_gas_consumption,natural_gas_reserves,coal_production,coal_consumption,coal_reserves
0,1,United States,20953000,20307000,38212000000,1029000000000,914301000000,13402000000000,534234000,495156000,247883000000
1,2,Saudi Arabia,11174000,3524000,258600000000,121870000000,121870000000,9423000000000,0,66000,0
2,3,Russia,10879000,3863000,80000000000,617830000000,472239000000,47805000000000,531130000,290763000,162166000000
3,4,Canada,5688000,2377000,170300000000,187686000000,130316000000,2067126000000,50687000,20092000,6582000000
4,5,China,4984000,16189000,26023000000,225341000000,366160000000,6654000000000,4805000000,5191000000,157041000000


## 11. Geography data collection

In [13]:
if RUN_SCRAPING:
    driver = create_driver()
    try:
        geography_urls = {
            "land_area_km2": "https://www.globalfirepower.com/square-land-area.php",
            **SOURCE_URLS["geography"],
        }

        df_geography_2026 = None
        for metric, url in geography_urls.items():
            metric_df = scrape_ranked_page(
                driver, url, metric, include_rank=(metric == "land_area_km2")
            )
            if df_geography_2026 is None:
                df_geography_2026 = metric_df
            else:
                df_geography_2026 = df_geography_2026.merge(metric_df, on="country", how="left")
    finally:
        close_driver(driver)
else:
    df_geography_2026 = pd.read_pickle(DATA_DIR / "df_geography_2026.pkl").copy()

df_geography_2026.head()

,rank,country,land_area_km2,coastline_coverage,border_coverage,waterway_coverage
0,1,Russia,17098242,37653,22407,102000
1,2,Canada,9984670,202080,8893,636
2,3,United States,9833517,19924,12002,41009
3,4,China,9596960,14500,22457,27700
4,5,Brazil,8515770,7491,16145,50000


## 12. Air Force data collection

In [14]:
if RUN_SCRAPING:
    driver = create_driver()
    try:
        airforce_urls = SOURCE_URLS["airforce"]

        df_airforce_2026 = None
        for metric, url in airforce_urls.items():
            metric_df = scrape_ranked_page(
                driver, url, metric, include_rank=(metric == "total_aircraft")
            )
            if df_airforce_2026 is None:
                df_airforce_2026 = metric_df
            else:
                df_airforce_2026 = df_airforce_2026.merge(metric_df, on="country", how="left")
    finally:
        close_driver(driver)
else:
    df_airforce_2026 = pd.read_pickle(DATA_DIR / "df_airforce_2026.pkl").copy()

df_airforce_2026.head()

,rank,country,total_aircraft,fighter_fleet,attack_fleet,transport_fleet,trainer_fleet,attack_helicopter_fleet,tanker_fleet,helicopter_fleet
0,1,United States,13032,1791,926,917,2610,1024,610,5913
1,2,Russia,4237,861,698,458,530,556,18,1643
2,3,China,3529,1443,371,287,401,281,9,1007
3,4,India,2183,476,124,277,334,79,6,594
4,5,South Korea,1540,242,98,40,335,113,4,827


## 13. Army data collection

In [15]:
if RUN_SCRAPING:
    driver = create_driver()
    try:
        army_urls = SOURCE_URLS["army"]

        df_army_2026 = None
        for metric, url in army_urls.items():
            metric_df = scrape_ranked_page(
                driver, url, metric, include_rank=(metric == "tank_fleet_strength")
            )
            if df_army_2026 is None:
                df_army_2026 = metric_df
            else:
                df_army_2026 = df_army_2026.merge(metric_df, on="country", how="left")
    finally:
        close_driver(driver)
else:
    df_army_2026 = pd.read_pickle(DATA_DIR / "df_army_2026.pkl").copy()

df_army_2026.head()

,rank,country,tank_fleet_strength,armored_fighting_vehicle,self_propelled_artillery,towed_artillery,rocket_projectors
0,1,China,5870,152040,2940,1400,2770
1,2,Russia,5630,126512,3603,5920,2486
2,3,North Korea,4895,47792,1300,700,1500
3,4,United States,4666,409660,1521,1878,1731
4,5,India,3913,163554,100,5640,300


## 14. Navy data collection

In [16]:
if RUN_SCRAPING:
    driver = create_driver()
    try:
        navy_urls = SOURCE_URLS["navy"]

        df_navy_2026 = None
        for metric, url in navy_urls.items():
            metric_df = scrape_ranked_page(
                driver, url, metric, include_rank=(metric == "naval_fleet_strength")
            )
            if df_navy_2026 is None:
                df_navy_2026 = metric_df
            else:
                df_navy_2026 = df_navy_2026.merge(metric_df, on="country", how="left")
    finally:
        close_driver(driver)
else:
    df_navy_2026 = pd.read_pickle(DATA_DIR / "df_navy_2026.pkl").copy()

df_navy_2026.head()

,rank,country,naval_fleet_strength,navy_force_by_tonnage,aircraft_carriers,helicopter_carriers,submarines,destroyers,frigates,corvettes,coastal_patrol,mine_warfare
0,1,China,841,3192411.0,3,4,61,53,46,50,150,36
1,2,Russia,747,1426539.0,1,0,66,13,12,79,70,45
2,3,United States,465,8265799.0,11,9,66,83,0,27,0,4
3,4,India,343,631989.0,2,0,18,13,18,21,146,0
4,5,Indonesia,338,324754.0,0,0,4,0,10,26,211,10


## 15. Data transformation and standardization

The original project transformed large values into analysis-friendly units. The final snapshot uses:

- manpower/population: **millions**
- defense budget/debt/reserves: **billions**
- PPP: **trillions**
- oil production/consumption: **million barrels per day**
- oil reserves: **billion barrels**
- natural gas: **trillion cubic metres**
- coal production/consumption: **million tonnes**
- coal reserves: **billion tonnes**

Column names are standardized to lowercase `snake_case` for consistent downstream analysis.

In [17]:
def standardize_columns(df):
    """Convert column names to lowercase snake_case."""
    result = df.copy()
    result.columns = (
        result.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return result


if RUN_SCRAPING:
    # The live scrape returns absolute manpower counts. Convert them to the
    # same million-unit convention used by the submitted project snapshot.
    manpower_scrape_to_millions = {
        "available_manpower_million": 1e6,
        "paramilitary_million": 1e6,
        "labor_force_strength_million": 1e6,
        "active_manpower_million": 1e6,
        "active_reserves_million": 1e6,
        "fit_for_service_million": 1e6,
    }

    for col, divisor in manpower_scrape_to_millions.items():
        if col in df_manpower_2026.columns:
            df_manpower_2026[col] = (df_manpower_2026[col] / divisor).round(3)

    # Finance
    for col, divisor, decimals in [
        ("annual_defense_budget", 1e9, 2),
        ("external_debt", 1e9, 2),
        ("purchasing_power_parity", 1e12, 2),
        ("foreign_exchange_reserves", 1e9, 2),
    ]:
        if col in df_finance_2026.columns:
            df_finance_2026[col] = (df_finance_2026[col] / divisor).round(decimals)

    # Natural resources
    for col, divisor, decimals in [
        ("oil_production", 1e6, 2),
        ("oil_consumption", 1e6, 2),
        ("oil_reserves", 1e9, 2),
        ("natural_gas_production", 1e12, 2),
        ("natural_gas_consumption", 1e12, 2),
        ("natural_gas_reserves", 1e12, 2),
        ("coal_production", 1e6, 2),
        ("coal_consumption", 1e6, 2),
        ("coal_reserves", 1e9, 2),
    ]:
        if col in df_nat_resources_2026.columns:
            df_nat_resources_2026[col] = (
                df_nat_resources_2026[col] / divisor
            ).round(decimals)

# Standardize schemas in both review and collection modes.
for name, data in list({
    "df_manpower_2026": df_manpower_2026,
    "df_finance_2026": df_finance_2026,
    "df_logistics_2026": df_logistics_2026,
    "df_geography_2026": df_geography_2026,
    "df_nat_resources_2026": df_nat_resources_2026,
    "df_airforce_2026": df_airforce_2026,
    "df_army_2026": df_army_2026,
    "df_navy_2026": df_navy_2026,
}.items()):
    globals()[name] = standardize_columns(data)

print("Transformation and schema standardization complete.")

Transformation and schema standardization complete.


## 16. Data-quality validation

Before exporting the analysis-ready files, the project validates the common country key and checks for duplicate records, missing values and dataset dimensions.

In [18]:
prepared_datasets = {
    "df_manpower_2026": df_manpower_2026,
    "df_finance_2026": df_finance_2026,
    "df_airforce_2026": df_airforce_2026,
    "df_army_2026": df_army_2026,
    "df_navy_2026": df_navy_2026,
    "df_logistics_2026": df_logistics_2026,
    "df_geography_2026": df_geography_2026,
    "df_nat_resources_2026": df_nat_resources_2026,
}

validation_rows = []

for name, data in prepared_datasets.items():
    validation_rows.append({
        "dataset": name,
        "rows": len(data),
        "columns": len(data.columns),
        "unique_countries": data["country"].nunique(),
        "duplicate_country_rows": data["country"].duplicated().sum(),
        "missing_cells": int(data.isna().sum().sum()),
    })

validation_summary = pd.DataFrame(validation_rows)
validation_summary

,dataset,rows,columns,unique_countries,duplicate_country_rows,missing_cells
0,df_manpower_2026,145,9,145,0,0
1,df_finance_2026,145,6,145,0,0
2,df_airforce_2026,145,10,145,0,0
3,df_army_2026,145,7,145,0,0
4,df_navy_2026,145,12,145,0,94
5,df_logistics_2026,145,7,145,0,0
6,df_geography_2026,145,6,145,0,0
7,df_nat_resources_2026,145,11,145,0,0


In [28]:
# Confirm that the country universe is consistent across datasets.
country_sets = {
    name: set(data["country"])
    for name, data in prepared_datasets.items()
}

reference_countries = country_sets["df_manpower_2026"]

country_consistency = pd.DataFrame([
    {
        "dataset": name,
        "matches_manpower_country_set": countries == reference_countries,
        "countries_not_in_manpower": len(countries - reference_countries),
        "countries_missing_from_dataset": len(reference_countries - countries),
    }
    for name, countries in country_sets.items()
])

country_consistency

,dataset,matches_manpower_country_set,countries_not_in_manpower,countries_missing_from_dataset
0,df_manpower_2026,True,0,0
1,df_finance_2026,True,0,0
2,df_airforce_2026,True,0,0
3,df_army_2026,True,0,0
4,df_navy_2026,True,0,0
5,df_logistics_2026,True,0,0
6,df_geography_2026,True,0,0
7,df_nat_resources_2026,True,0,0


### Data Quality Note — Navy Fleet Tonnage

Fleet tonnage data is available for **51 of the 145 countries** in the Global Firepower dataset. For the remaining **94 countries**, the source does not provide fleet tonnage values.

These observations are therefore retained as **missing (`NaN`)** rather than being imputed as zero, since missing data does not imply zero fleet tonnage.

**Handling:** The Navy dataset retains all **145 countries**, while analyses specifically using fleet tonnage should be restricted to the **51 countries with available values**.

### Validation outcome

For the submitted project snapshot, the eight prepared datasets contain the same **145-country universe**, with one country-level record per dataset.

The `country` field is therefore the common analytical key used to integrate the datasets in the downstream EDA notebook.

## 17. Final dataset schemas

The final project snapshot contains eight analysis-ready datasets.

In [23]:
schema_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": data.shape[0],
        "columns": data.shape[1],
        "country_key": "country" in data.columns,
    }
    for name, data in prepared_datasets.items()
])

schema_summary

,dataset,rows,columns,country_key
0,df_manpower_2026,145,9,True
1,df_finance_2026,145,6,True
2,df_airforce_2026,145,10,True
3,df_army_2026,145,7,True
4,df_navy_2026,145,12,True
5,df_logistics_2026,145,7,True
6,df_geography_2026,145,6,True
7,df_nat_resources_2026,145,11,True


## 18. Export analysis-ready datasets

The `.pkl` files produced here are the handoff point between the **data preparation notebook** and the **EDA / Business Analysis notebook**.

In [24]:
# Save only when the notebook has collected/transformed the datasets.
# In review mode (RUN_SCRAPING=False), the existing submitted PKLs are not overwritten.

if RUN_SCRAPING:
    for name, data in prepared_datasets.items():
        data.to_pickle(DATA_DIR / f"{name}.pkl")
    print(f"Saved {len(prepared_datasets)} analysis-ready datasets.")
else:
    print("Review mode: existing project PKL files were preserved.")

Saved 8 analysis-ready datasets.


## 19. Final handoff to EDA

At this stage the data pipeline is complete.

### Output datasets

```text
df_manpower_2026.pkl
df_finance_2026.pkl
df_airforce_2026.pkl
df_army_2026.pkl
df_navy_2026.pkl
df_logistics_2026.pkl
df_geography_2026.pkl
df_nat_resources_2026.pkl
```

The next notebook, **`Global Firepower EDA.ipynb`**, uses these prepared datasets to:

1. integrate the eight dimensions,
2. explore relationships and distributions,
3. engineer analytical metrics,
4. evaluate the project's business questions, and
5. communicate the resulting insights.

This separation keeps the project flow clear:

> **Collection & preparation → EDA → business analysis → insights**.